# 📊 Project 5: Financial Sentiment Analysis from News & Social Media


## 📌 Objective:
Build an NLP pipeline to extract, analyze, and visualize sentiment from financial news articles and social media (e.g., Twitter, Reddit) related to stocks or market trends.



## 🔧 Tools & Libraries:
- `pandas`, `matplotlib`, `seaborn` – Data handling and visualization
- `requests`, `snscrape` – Data scraping
- `transformers`, `torch`, `textblob`, `vaderSentiment` – Sentiment analysis
- `wordcloud`, `sklearn` – Visualization & modeling


In [ ]:

# ✅ Install necessary packages (Uncomment if running in a new environment)
# !pip install pandas matplotlib seaborn snscrape transformers torch textblob vaderSentiment wordcloud scikit-learn


In [ ]:

# 📥 Step 1: Collect Financial News Headlines
import requests
import pandas as pd
from bs4 import BeautifulSoup

def fetch_financial_news():
    url = "https://www.financialexpress.com/market/"
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    headlines = [headline.get_text().strip() for headline in soup.find_all('h3')]
    return pd.DataFrame({'headline': headlines})

df_news = fetch_financial_news()
df_news.head()


In [ ]:

# 🐦 Step 2: Scrape Tweets using snscrape
import snscrape.modules.twitter as sntwitter

query = "stock market since:2023-01-01 until:2023-12-31"
tweets = []
for i, tweet in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
    if i > 100:
        break
    tweets.append([tweet.date, tweet.user.username, tweet.content])

df_tweets = pd.DataFrame(tweets, columns=['date', 'user', 'text'])
df_tweets.head()


In [ ]:

# 🔍 Step 3: Sentiment Analysis using VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

df_news['sentiment'] = df_news['headline'].apply(analyze_sentiment)
df_tweets['sentiment'] = df_tweets['text'].apply(analyze_sentiment)

df_news.head()


In [ ]:

# 📊 Step 4: Visualize Sentiment Distributions
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))
sns.histplot(df_news['sentiment'], kde=True, color='blue', label='News')
sns.histplot(df_tweets['sentiment'], kde=True, color='orange', label='Tweets')
plt.title("Sentiment Distribution: Financial News vs Tweets")
plt.legend()
plt.show()


In [ ]:

# ☁️ Step 5: Generate Word Clouds
from wordcloud import WordCloud

wordcloud_news = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df_news['headline']))
wordcloud_tweets = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df_tweets['text']))

plt.figure(figsize=(14,6))
plt.subplot(1, 2, 1)
plt.imshow(wordcloud_news, interpolation='bilinear')
plt.title("News Headlines WordCloud")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(wordcloud_tweets, interpolation='bilinear')
plt.title("Tweets WordCloud")
plt.axis("off")
plt.show()
